# **作业1：Hessian矩阵**

幻灯片: https://speech.ee.ntu.edu.tw/~hylee/ml/ml2021-course-data/hw/HW02/HW02


## Hessian矩阵

作业中判断模型是否处于“类局部极小值点（Local Minima-like Point）”的具体量化标准。   

由于在真实的浮点数计算机运算中，我们很难遇到严格为 0 的梯度或绝对完美的 100% 正特征值，因此作业给出了两个工程上的近似指标和判定阈值：

1. 梯度范数（Gradient Norm）
    - 含义：用来衡量当前点处的梯度向量有多大（即陡不陡峭）。
    - 工程处理：在实际训练中，梯度几乎不可能刚好精确等于 0。因此作业规定：当梯度的范数（L2 Norm）小于 1e-3（即 $< 0.001$）时，我们就认为此时梯度已经“接近于零”，模型进入了驻点/临界点区域。
2. 极小值比例（Minimum Ratio）
    - 定义公式：$$\text{Minimum ratio} = \frac{\text{正特征值的数量}}{\text{总特征值的数量}}$$ 即 Hessian 矩阵中大于 0 的特征值所占的比例。  
    - 数学含义：理想的局部极小值（Strict Local Minima）：所有特征值都为正（$\text{Minimum ratio} = 1.0$）。   
    - 鞍点（Saddle Point）：特征值有正有负（$\text{Minimum ratio} < 1.0$）。
3. 本作业的具体判定法则（Decision Rule）
    - PPT 中用红字标注了最终的判定条件：$$\text{若 } \mathbf{\text{Minimum ratio} > 0.5} \quad \text{且} \quad \mathbf{\text{Gradient Norm} < 1\text{e}-3}$$只要满足这两个条件，作业就认定模型当前处于 “类局部极小值（local minima like）” 的位置！
4. 为什么设为 Minimum ratio > 0.5？
    - 在深度学习的高维参数空间中，极小值点（要求 100% 正特征值）极其罕见，绝大多数驻点都是 Saddle Point（鞍点）。

## 重要提示
1. 结果因人而异：在本次作业中，输入不同的学号（Student ID）会得到不同的计算结果和标准答案。
2. 正确填写的必要性：必须在指定代码块中填入正确的学号，否则代码可能无法正常运行或导致答案错误。
3. 助教把 student_id 字符串当作随机种子（Seed），用来动态生成或分配专属的模型 Checkpoint。这样每个校内学生的作业数据和输出结果都不一样，防止直接抄袭别人的答案。
4. 选定后保持固定：在完成整个实验的过程中，不要再去修改这个字符串。这样每次运行加载的都是同一个模型状态，你计算出的 Gradient Norm、Minimum Ratio 以及形态判断结果就会始终保持自洽。

In [37]:
student_id = 'SunshineZ' # fill with your student ID

assert student_id != 'your_student_id', 'Please fill in your student_id before you start.'

## 计算Hessian矩阵
Hessian矩阵的计算由TA完成，您无需也不应修改以下代码。您只需运行以下代码块，并根据梯度范数和最小比值的数值判断模型处于“局部极小点”、“鞍点”或“以上都不是”状态即可。

### 安装计算 Hessian 矩阵的核心第三方工具包

autograd-lib 库用于计算Hessian矩阵。您可在此处查看完整文档：https://github.com/cybertronai/autograd-lib。

In [38]:
!pip install autograd-lib

### 导入必要Python库


In [39]:
import numpy as np
from math import pi
from collections import defaultdict
from autograd_lib import autograd_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import warnings
warnings.filterwarnings("ignore")

### 定义神经网络模型
此处的神经网络模型用于拟合单变量数学函数：
$$f(x) = \frac{\sin(5\pi x)}{5\pi x}.$$

In [40]:
class MathRegressor(nn.Module):
    def __init__(self, num_hidden=128):
        super().__init__()
        self.regressor = nn.Sequential(
            nn.Linear(1, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, 1)
        )

    def forward(self, x):
        x = self.regressor(x)
        return x

### 获取预训练检查点
模型检查点（checkpoint）是助教（TA）预先训练好的，每个学生拿到的都不一样。

In [41]:
!gdown --id 1ym6G7KKNkbsqSnMmnxdQKHO1JBoF0LPR

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1ym6G7KKNkbsqSnMmnxdQKHO1JBoF0LPR
To: /content/data.pth
100% 34.5k/34.5k [00:00<00:00, 47.4MB/s]


### 加载预训练检查点和训练数据

In [42]:
# find the key from student_id
import re

key = student_id[-1]
if re.match('[0-9]', key) is not None:
    key = int(key)
else:
    key = ord(key) % 10

In [43]:
# load checkpoint and data corresponding to the key
model = MathRegressor()
autograd_lib.register(model)

data = torch.load('data.pth')[key]
model.load_state_dict(data['model'])
train, target = data['data']

### 计算模型梯度范数（Gradient Norm）的函数实现

In [44]:
def compute_gradient_norm(model, criterion, train, target):
    # 切换为训练模式
    model.train()
    # 清空模型之前的历史梯度
    model.zero_grad()
    # 前向传播：计算模型对输入的预测值
    output = model(train)
    # 计算损失值 (Loss)
    loss = criterion(output, target)
    # 反向传播：计算每个参数的梯度
    loss.backward()

    # 每一层梯度s
    grads = []
    # 遍历 regressor 这个子模块的直接子层（通常是 nn.Sequential 里的 Linear 和激活函数）。
    for p in model.regressor.children():
        # 只挑选 nn.Linear 层，取该层权重矩阵的梯度。
        if isinstance(p, nn.Linear):
            # 计算该梯度矩阵的 L2 范数
            # p.weight：存放的是权重参数本身（Weight Values）。二维矩阵
            # p.weight.grad：存放的是针对该权重的梯度值（Gradients of the Weights）。F范数：全部平方、求和、再开平方根
            # item()，把 PyTorch 张量转成 Python 浮点数。
            param_norm = p.weight.grad.norm(2).item()
            # 把每层的梯度范数加入列表 grads
            grads.append(param_norm)

    # 求列表 grads 的平均值
    grad_mean = np.mean(grads)

    return grad_mean

### 计算 Minimum Ratio（最小比例）函数实现

In [45]:
# source code from the official document https://github.com/cybertronai/autograd-lib

# hook，在前向传播时，把每一层网络收到的“输入数据（激活值 A）”拦截并保存到字典里
def save_activations(layer, A, _):
    '''
    A is the input of the layer, we use batch size of 6 here
    layer 1: A has size of (6, 1)
    layer 2: A has size of (6, 128)
    '''
    activations[layer] = A

# hook，利用神经网络中各层的一阶梯度（通过反向传播误差与激活值的组合），在 Batch（批次）维度上进行外积并累加，从而“近似”计算出该层参数的 Hessian 矩阵（二阶导数矩阵）
def compute_hess(layer, _, B):
    '''
    B is the backprop value of the layer
    layer 1: B has size of (6, 128)
    layer 2: B ahs size of (6, 1)
    '''
    A = activations[layer]
    BA = torch.einsum('nl,ni->nli', B, A) # do batch-wise outer product

    # full Hessian
    hess[layer] += torch.einsum('nli,nkj->likj', BA, BA) # do batch-wise outer product, then sum over the batch

补充：
1. 第一层：
  - 有 128 个神经元（$L = 128$），每个神经元有 1 个权重（$I = 1$）。
  - 这一层的总参数量是：$128 \times 1 = 128$ 个。
  - 对应的完整 2D Hessian 矩阵大小就是 $(128, 128)$。
2. 第二层：
  - 有 1 个神经元（$L = 1$），这个神经元连接了上一层的 128 个输入（$I = 128$）。- 这一层的总参数量同样是：$1 \times 128 = 128$ 个。
  - 对应的完整 2D Hessian 矩阵大小也是 $(128, 128)$！
3. $$H =
\begin{bmatrix}
g_1^2 & g_1g_2 & \dots & g_1g_{128} \\
g_2g_1 & g_2^2 & \dots & g_2g_{128} \\
\vdots & \vdots & \ddots & \vdots \\
g_{128}g_1 & g_{128}g_2 & \dots & g_{128}^2
\end{bmatrix}$$
  - $g_N$ 表示这一层第 $N$ 个权重参数的梯度值，是一个标量。$$g_i = \frac{\partial L}{\partial w_i}$$
4. hess[layer]，形状 (1, 128, 1, 128)
5. 如果一个 $n \times n$ 的矩阵拥有足够多的线性无关特征向量，并且这些特征向量能够张成全空间（即包含 $n$ 个线性无关的特征向量），那么它就一定可以被对角化。也就是说，必定存在可逆矩阵 $P$，使得：$$D = P^{-1} A P$$（或者写成 $A = P D P^{-1}$）这里的 $D$ 就是一个对角矩阵，其主对角线上的元素正是该矩阵的特征值。
6. | 所有特征值的正负组合 | 局部几何形态 | 弯曲特征描述 | 极值结论 |
| :--- | :--- | :--- | :--- |
| **全部 $\lambda_i > 0$** | **碗状凹陷（Bowl shape）** | 所有主轴方向均向上弯曲升高 | **严格局部极小值** |
| **全部 $\lambda_i < 0$** | **倒扣的碗（Cap shape）** | 所有主轴方向均向下弯曲降低 | **严格局部极大值** |
| **特征值有正有负** | **马鞍面（Saddle shape）** | 某些方向向上弯曲，某些方向向下弯曲 | **鞍点（不是极值点）** |
| **存在 $\lambda_i = 0$（其余非负/非正）** | **退化/退化槽面（Degenerate）** | 沿着零特征值对应的方向，地形不发生变化 | **无法单纯凭二阶信息判定（退化点）** |

In [46]:
# 计算模型在所有线性层上的“正特征值比例”（minimum ratio），作为判断当前训练点是否接近局部极小点的指标
def compute_minimum_ratio(model, criterion, train, target):
    # 清空梯度，准备前向传播
    model.zero_grad()
    # 在前向传播时保存每一层的输入 A
    with autograd_lib.module_hook(save_activations):
        output = model(train)
        loss = criterion(output, target)

    # 触发一次“专门算二阶导的反向传播”
    # 反向走一遍网络，在每一层触发 compute_hess，把该层的近似 Hessian 累加到 hess[layer]
    with autograd_lib.module_hook(compute_hess):
        autograd_lib.backward_hessian(output, loss='LeastSquares')


    '''
    hess = {
      第一层: tensor(128,1,128,1),
      第二层: tensor(1,128,1,128)
    }
    layer_hess = [
      第一层的 Hessian 矩阵 (4D 张量，形状是 128, 1, 128, 1),
      第二层的 Hessian 矩阵 (4D 张量，形状是 1, 128, 1, 128)
    ]
    '''
    layer_hess = list(hess.values())
    minimum_ratio = []

    # 遍历每一层的 Hessian 张量
    for h in layer_hess:
        # 每一层的参数总数是
        size = h.shape[0] * h.shape[1]
        # 通过 reshape(size, size)，把那个让人头大的 4D 网格张量，拍扁成你推导出的标准的 (128, 128) 二维方阵。
        h = h.reshape(size, size)
        # 对这个对称矩阵求特征值。torch.symeig 返回特征值和特征向量，这里只要特征值。
        # h_eig = torch.symeig(h).eigenvalues
        # 对这个对称矩阵求特征值，并加上微小的单位矩阵扰动以防病态矩阵不收敛
        h = h + torch.eye(h.size(0), device=h.device) * 1e-5
        h_eig = torch.linalg.eigvalsh(h)
        # 统计正特征值的个数。
        num_greater = torch.sum(h_eig > 0).item()
        # 计算正特征值占总特征值的比例，存入 minimum_ratio 列表。
        minimum_ratio.append(num_greater / len(h_eig))

    # 循环结束后，minimum_ratio 里就有每一层的比例。后面代码会取平均。
    ratio_mean = np.mean(minimum_ratio)

    return ratio_mean

### 为什么代码里可以用“梯度外积”来偷懒代替复杂的 Hessian 矩阵

这个方法在机器学习中被称为 Gauss-Newton（高斯-牛顿）近似算法。

1. Hessian 矩阵的本来面目（真实的二阶导数），公式开头发问：Hessian 矩阵的每一个元素 $\mathbf{H}_{ij}$ 到底是什么？
  - 它是损失函数 $L$ 对两个权重 $w_i$ 和 $w_j$ 的二阶导数：$$\mathbf{H}_{ij} = \frac{\partial^2 L}{\partial w_i \partial w_j}$$
  - 假设我们用最简单的均方误差 $L = e^2$（其中 $e$ 是误差），对它求两次导，展开后会得到两项：$$\mathbf{H}_{ij} = 2\frac{\partial e}{\partial w_i}\frac{\partial e}{\partial w_j} + 2e\frac{\partial^2 e}{\partial w_j \partial w_i}$$

2. 为什么要“偷懒”（高斯-牛顿假设）？在上面那堆式子里，第二项长这样：$2e\frac{\partial^2 e}{\partial w_j \partial w_i}$。
  - 这项里面包含了误差 $e$ 和 误差的二阶导数。
  - 为什么要扔掉它？
    - 如果网络训练得差不多了，误差 $e$ 本身就很小，乘以 $e$ 之后这一项几乎就等于 0 了。
    - 求真正的二阶导数（误差的二阶导）计算量实在太恐怖了，计算机根本算不动。
  - 于是，高斯-牛顿算法大笔一挥：把复杂的第二项直接忽略/抹掉（Neglect）！

3. 简化后的结果：只剩一阶导数相乘，扔掉第二项之后，Hessian 矩阵就被大大简化了，它只正比例于第一项：$$\mathbf{H}_{ij} \propto \frac{\partial e}{\partial w_i}\frac{\partial e}{\partial w_j}$$而误差的一阶导数，本质上就对应着我们平时用的梯度（损失对权重的导数 $\frac{\partial L}{\partial w}$）。

4. 最终的结论，所以，最后推导出来的结论是：$$\mathbf{H} \propto (\mathbf{B} \times \mathbf{A}) \times (\mathbf{B} \times \mathbf{A})$$
  - 这里的 $\mathbf{B} \times \mathbf{A}$ 就是我们前面提到的该层梯度（由反向传播值 $\mathbf{B}$ 和输入 $\mathbf{A}$ 组合而成）。
  - 整段话的意思就是：我们不需要去费劲计算真实的、复杂的二阶导数 Hessian 矩阵；我们只需要用梯度（$\mathbf{B}$ 和 $\mathbf{A}$）自己跟自己做外积，就能近似模拟出 Hessian 矩阵了！


In [47]:
# 取得梯度范数和极小值比例
def main(model, train, target):
    criterion = nn.MSELoss()

    gradient_norm = compute_gradient_norm(model, criterion, train, target)
    minimum_ratio = compute_minimum_ratio(model, criterion, train, target)

    print('gradient norm: {}, minimum ratio: {}'.format(gradient_norm, minimum_ratio))

    # 判断状态
    if gradient_norm >= 1e-3:
        print("none of the above（梯度还大）")
    elif minimum_ratio > 0.5:
        print("local minima like（局部极小值）")
    else:
        print("saddle point（鞍点）")

根据计算出的 `minimum ratio`（以及 `gradient norm`），判断当前模型处于哪种状态，然后把你判断的结果提交到 NTU COOL 作业系统里。

In [48]:
if __name__ == '__main__':
    # fix random seed
    torch.manual_seed(0)

    # reset compute dictionaries
    activations = defaultdict(int)
    hess = defaultdict(float)

    # compute Hessian
    main(model, train, target)

gradient norm: 0.0009449839126318693, minimum ratio: 1.0
local minima like（局部极小值）
